<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/similarity_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [125]:
%pip -q install pandas numpy scipy scikit-learn plotly

In [126]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews_preprocessed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

preprocessed_reviews_df = pd.DataFrame(data)

In [127]:
preprocessed_reviews_df.head()

,review_id,review_date,place_id,review_text,rating,description_clean,description_tokenized,description_normalized,description_no_stopwords,description_stemmed,description_lemmatized,description_bigrams
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,2026-07-30T17:09:02+00:00,0x6b0a883715bb4c6b:0,Excelente restaurante! A comida é muito saboro...,5.0,Excelente restaurante! A comida é muito saboro...,"[Excelente, restaurante, !, A, comida, é, muit...","[excelente, restaurante, a, comida, é, muito, ...","[excelente, restaurante, comida, muito, saboro...","[excel, restaurant, com, muit, sabor, bem, ser...","[excelente, restaurante, comer, muito, saboros...","[[excelente, restaurante], [restaurante, comer..."
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Fui pela primeira vez e gostei bastante! A com...,5.0,Fui pela primeira vez e gostei bastante! A com...,"[Fui, pela, primeira, vez, e, gostei, bastante...","[fui, pela, primeira, vez, e, gostei, bastante...","[primeira, vez, gostei, bastante, comida, muit...","[prim, vez, gost, bast, com, muit, sabor, peg,...","[primeiro, vez, gostar, bastante, comer, muito...","[[primeiro, vez], [vez, gostar], [gostar, bast..."
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Ambiente simples e gostoso familiar comida óti...,5.0,Ambiente simples e gostoso familiar comida óti...,"[Ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, gostoso, familiar, comida,...","[ambi, simpl, gost, famili, com, ótim, gost, m...","[ambiente, simples, gostoso, familiar, comer, ...","[[ambiente, simples], [simples, gostoso], [gos..."
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,2022-08-30T17:10:53+00:00,0x6b0a883715bb4c6b:0,"Se é qualidade que procura, não tem outra igua...",5.0,"Se é qualidade que procura, não tem outra igua...","[Se, é, qualidade, que, procura, ,, não, tem, ...","[se, é, qualidade, que, procura, não, tem, out...","[qualidade, procura, não, outra, igual, agilid...","[qual, proc, não, outr, igual, agil, entreg, t...","[qualidade, procurar, não, outro, igual, agili...","[[qualidade, procurar], [procurar, não], [não,..."
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,2022-08-30T17:10:52+00:00,0x6b0a883715bb4c6b:0,"Comida maravilhosa. Super caseira, bem tempera...",5.0,"Comida maravilhosa. Super caseira, bem tempera...","[Comida, maravilhosa, ., Super, caseira, ,, be...","[comida, maravilhosa, super, caseira, bem, tem...","[comida, maravilhosa, super, caseira, bem, tem...","[com, maravilh, sup, cas, bem, temper, tend, s...","[comida, maravilhoso, super, caseira, bem, tem...","[[comida, maravilhoso], [maravilhoso, super], ..."


# Bag of Words

In [128]:
from sklearn.feature_extraction.text import CountVectorizer

# Extrai todos os tokens de todas as reviews
tokens_column = preprocessed_reviews_df["description_no_stopwords"]
tokens = tokens_column.tolist()

bow_vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens
)
bow_matrix = bow_vectorizer.fit_transform(
    tokens_column
)

# Recupera o vocabulário
bow_words = bow_vectorizer.get_feature_names_out()

# Transforma a matriz em DataFrame
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_words
)

# Adicionar o ID para conectar com a base original
df_bow.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

# Visualizar
df_bow.head()

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fui pela primeira vez e gostei bastante! A com...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ambiente simples e gostoso familiar comida óti...,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,"Se é qualidade que procura, não tem outra igua...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"Comida maravilhosa. Super caseira, bem tempera...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# TF-IDF

In [129]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criar o modelo TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

# Criar matriz
tfidf_matrix = tfidf_vectorizer.fit_transform(
    preprocessed_reviews_df["description_no_stopwords"]
)

# Recuperar o vocabulário
tfidf_words = tfidf_vectorizer.get_feature_names_out()

# Transformar matriz em DF
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_words
)

df_tfidf.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

df_tfidf.head(20)

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,Fui pela primeira vez e gostei bastante! A com...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,Ambiente simples e gostoso familiar comida óti...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.151180,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,"Se é qualidade que procura, não tem outra igua...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,"Comida maravilhosa. Super caseira, bem tempera...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
5,As palavras para descrever a comida é simplesm...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
6,"A comida deve ser boa pelas indicações mas, o ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7,Excelente comida e serviço,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
8,Top,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
9,Não sei na outras empresas mas já foi servido...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


# Similaridade entre textos

In [130]:
from sklearn.metrics.pairwise import cosine_similarity

In [131]:
# Calcula similaridade a partir do BoW
similarity_bow = cosine_similarity(
    df_bow.drop(columns="review_text")
)

dt_sim_bow = pd.DataFrame(
    similarity_bow,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_bow.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.227110,0.183533,0.223607,0.127920,0.300000,0.051640,0.230940,0.0,0.044721,...,0.424264,0.2,0.0,0.0,0.2,0.0,0.000000,0.200000,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.227110,1.000000,0.186081,0.108821,0.138343,0.189258,0.041885,0.093659,0.0,0.072548,...,0.229416,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.093659
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.183533,0.186081,1.000000,0.153897,0.097823,0.191180,0.059235,0.132453,0.0,0.051299,...,0.162221,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.223607,0.108821,0.153897,1.000000,0.000000,0.372678,0.000000,0.000000,0.0,0.100000,...,0.316228,0.0,0.0,0.0,0.0,0.0,0.000000,0.111803,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.127920,0.138343,0.097823,0.000000,1.000000,0.106600,0.110096,0.246183,0.0,0.095346,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.123091,0.000000,0.0,0.369274


In [132]:
# Calcula similaridade a partir do TF-IDF
similarity_tfidf = cosine_similarity(
    df_tfidf.drop(columns="review_text")
)

dt_sim_tfidf = pd.DataFrame(
    similarity_tfidf,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_tfidf.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.101567,0.046544,0.074669,0.041217,0.123324,0.008559,0.087817,0.0,0.007912,...,0.261253,0.122276,0.0,0.0,0.122276,0.0,0.000000,0.101837,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.101567,1.000000,0.080314,0.035230,0.045367,0.080605,0.005258,0.020171,0.0,0.024956,...,0.125176,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.052753
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.046544,0.080314,1.000000,0.063365,0.015287,0.055853,0.007381,0.028316,0.0,0.006823,...,0.049590,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.074669,0.035230,0.063365,1.000000,0.000000,0.185659,0.000000,0.000000,0.0,0.049793,...,0.117975,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.107273,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.041217,0.045367,0.015287,0.000000,1.000000,0.031610,0.016993,0.065189,0.0,0.015709,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.070189,0.000

# Heatmap de similaridade

In [133]:
import plotly.graph_objects as go

texts = preprocessed_reviews_df["review_text"]

def create_heatmap(matrix, title):
  custom_data = []

  for i in range(len(texts)):
    line = []

    for j in range(len(texts)):
      line.append([
          texts[i],
          texts[j]
      ])

    custom_data.append(line)

  fig = go.Figure(
      data=go.Heatmap(
          z=matrix,
          x=texts,
          y=texts,
          colorscale="Blues",
          zmin=0,
          zmax=1,
          customdata=custom_data,
          hovertemplate=
            "<b>%{customdata[0]}</b><br>"
                "<b>%{customdata[1]}</b><br>"
                "<b>Similaridade:</b> %{z:.3f}"
                "<extra></extra>"
      )
  )

  fig.update_layout(
      title=title,
      xaxis_title="Review",
      yaxis_title="Review",
      width=900,
      height=850
  )

  fig.show()

In [ ]:
create_heatmap(
    similarity_bow,
    "Similaridade entre reviews — Bag of Words"
)

In [ ]:
create_heatmap(
    similarity_tfidf,
    "Similaridade entre reviews — TF-IDF"
)

# Clustering

In [135]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.metrics import silhouette_score

# Número de amostras
n_samples = tfidf_matrix.shape[0]

# Valores de K testados
k_values = [
    k for k in range(2, min(11, n_samples))
]

results = []
models = {}

# Criar e ajustar o modelo
for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
        max_iter=300
    )

    labels = model.fit_predict(tfidf_matrix)
    n_labels = len(np.unique(labels))

    if 1 < n_labels < n_samples:
        silhouette = silhouette_score(
            tfidf_matrix,
            labels,
            metric="cosine",
            sample_size=min(2000, n_samples),
            random_state=42
        )
    else:
        silhouette = np.nan

    results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_cosine": silhouette,
        "cluster_sizes": np.bincount(labels).tolist()
    })

    models[k] = model

# Adicionar o resultado a um dataframe
metrics_df = pd.DataFrame(results)

In [136]:
import plotly.express as px

# Visualizar as métricas
fig = px.line(
    metrics_df,
    x="k",
    y="inertia",
    markers=True,
    title="Método do cotovelo (TF-IDF + K-Means)",
    labels={
        "k": "Número de clusters (K)",
        "inertia": "Inércia"
    }
)

fig.show()

In [137]:
display(
    metrics_df.sort_values(
        "silhouette_cosine",
        ascending=False
    )
)

,k,inertia,silhouette_cosine,cluster_sizes
8,10,1733.299748,0.160650,"[111, 160, 1337, 69, 153, 92, 113, 104, 36, 87]"
6,8,1807.808376,0.142176,"[100, 105, 164, 126, 1527, 154, 69, 17]"
7,9,1780.215481,0.139931,"[40, 1554, 112, 160, 93, 113, 49, 35, 106]"
5,7,1829.599049,0.125999,"[116, 91, 123, 106, 163, 93, 1570]"
4,6,1853.751729,0.117366,"[1665, 124, 106, 116, 88, 163]"
3,5,1926.594322,0.105120,"[166, 277, 107, 94, 1618]"
2,4,1938.903438,0.095556,"[129, 1856, 106, 171]"
1,3,2006.829995,0.082741,"[170, 1985, 107]"
0,2,2090.226321,0.047895,"[2092, 170]"


In [142]:
# Selecionar o melhor modelo

N_CLUSTERS = 10
kmeans = models[N_CLUSTERS]
preprocessed_reviews_df["cluster"] = kmeans.labels_

# Análise de clustering


## Verificação manual


In [143]:
# Verificar 10 amostras de cada cluster
for cluster_id in sorted(preprocessed_reviews_df["cluster"].unique()):
    cluster_df = preprocessed_reviews_df[preprocessed_reviews_df["cluster"] == cluster_id]

    print(f"\n{'=' * 80}")
    print(f"CLUSTER {cluster_id} — {len(cluster_df)} reviews")
    print("=" * 80)

    columns_to_show = [
        col for col in [
            "review_id",
            "place_id",
            "rating",
            "review_text"
        ]
        if col in cluster_df.columns
    ]

    display(cluster_df[columns_to_show].head(10))


CLUSTER 0 — 111 reviews


,review_id,place_id,rating,review_text
48,ChdDSUhNMG9nS0VJQ0FnSUREc3Z2Ni13RRAB,0x94df18bfe8d9f45f:0,5.0,Muito bom!
60,ChZDSUhNMG9nS0VJQ0FnSUNoeGZ5WWRnEAE,0x94df18bfe8d9f45f:0,5.0,Bom
139,ChdDSUhNMG9nS0VJQ0FnSURnMGZYa2tBRRAB,0x94df18bfe8d9f45f:0,5.0,Muito bom
152,ChdDSUhNMG9nS0VJQ0FnSUNwcTl5d3pnRRAB,0x94df18c387e4e047:0,5.0,Atendimento muito bom e comida deliciosa!!
171,ChZDSUhNMG9nS0VJQ0FnSUNJcWFHMlVBEAE,0x94df18c387e4e047:0,4.0,Bom
172,ChZDSUhNMG9nS0VJQ0FnSURJdlplOWRnEAE,0x94df18c387e4e047:0,4.0,Bom atendimento
184,ChZDSUhNMG9nS0VJQ0FnSUNvN2RHcURBEAE,0x94df18c51a132009:0,5.0,Muito bom
204,ChZDSUhNMG9nS0VJQ0FnSURwM2FXSEd3EAE,0x94df18c51a132009:0,3.0,Bom
239,ChdDSUhNMG9nS0VJQ0FnSUR5aU1qNGlRRRAB,0x94df18c51a132009:0,3.0,Bom
262,ChdDSUhNMG9nS0VJQ0FnSURZeUtYeS1RRRAB,0x94df18c51a132009:0,4.0,Bom



CLUSTER 1 — 160 reviews


,review_id,place_id,rating,review_text
7,ChZDSUhNMG9nS0VJQ0FnSURGaHJxYVNBEAE,0x94defd80e5de4f0f:0,5.0,Excelente comida e serviço
34,Ci9DQUlRQUNvZENodHljRjlvT2pKQmN6SkNlRkIzTTB0R1...,0x94df18bfe8d9f45f:0,5.0,Comida excelente
61,ChdDSUhNMG9nS0VJQ0FnSURoa2M3amtRRRAB,0x94df18bfe8d9f45f:0,5.0,Excelente
63,ChdDSUhNMG9nS0VJQ0FnSUR4MmJTbG5BRRAB,0x94df18bfe8d9f45f:0,5.0,excelente atendimento!
71,ChZDSUhNMG9nS0VJQ0FnSUM2bjhfdld3EAE,0x94df18bfe8d9f45f:0,5.0,Tudo excelente
73,ChZDSUhNMG9nS0VJQ0FnSUNXM0ozLVRnEAE,0x94df18bfe8d9f45f:0,5.0,Excelente
78,ChdDSUhNMG9nS0VJQ0FnSUNpczd6eTRRRRAB,0x94df18bfe8d9f45f:0,5.0,Excelente!
82,ChdDSUhNMG9nS0VJQ0FnSUN5cGZQdjZBRRAB,0x94df18bfe8d9f45f:0,5.0,Comida excelente!
89,ChZDSUhNMG9nS0VJQ0FnSUNjdy1LcFJnEAE,0x94df18bfe8d9f45f:0,5.0,O serviço é excelente
90,ChdDSUhNMG9nS0VJQ0FnSUNDN29fRDhBRRAB,0x94df18bfe8d9f45f:0,5.0,Excelente lugar



CLUSTER 2 — 1337 reviews


...

,review_id,place_id,rating,review_text
8,ChdDSUhNMG9nS0VJQ0FnSUNPd2UzSm9BRRAB,0x94defd80e5de4f0f:0,5.0,Top
12,ChdDSUhNMG9nS0VJQ0FnSUNhdnJTdl9nRRAB,0x94defd80e5de4f0f:0,5.0,Local top ótima comida
64,ChZDSUhNMG9nS0VJQ0FnSURHcDhiWElBEAE,0x94df18bfe8d9f45f:0,5.0,Top 👏👏 …
74,ChdDSUhNMG9nS0VJQ0FnSUNpbTQyaW9nRRAB,0x94df18bfe8d9f45f:0,5.0,Top
84,ChZDSUhNMG9nS0VJQ0FnSUQwcUlIREdREAE,0x94df18bfe8d9f45f:0,5.0,Top
97,ChdDSUhNMG9nS0VJQ0FnSUR3MklqTHFBRRAB,0x94df18bfe8d9f45f:0,5.0,Top
107,ChZDSUhNMG9nS0VJQ0FnSUNnMzdtVGZBEAE,0x94df18bfe8d9f45f:0,5.0,Top
169,ChdDSUhNMG9nS0VJQ0FnSUNJbnFuQ21nRRAB,0x94df18c387e4e047:0,5.0,Top
175,ChdDSUhNMG9nS0VJQ0FnSURnX19xeDNRRRAB,0x94df18c387e4e047:0,5.0,Atendimento Top!
214,ChdDSUhNMG9nS0VJQ0FnSUR4cUxiNG1RRRAB,0x94df18c51a132009:0,5.0,Top



CLUSTER 8 — 36 reviews


,review_id,place_id,rating,review_text
87,ChZDSUhNMG9nS0VJQ0FnSUNjcWRiVVlBEAE,0x94df18bfe8d9f45f:0,5.0,Comida sensacional.
124,ChdDSUhNMG9nS0VJQ0FnSURnMHNXbDRRRRAB,0x94df18bfe8d9f45f:0,5.0,"Sinceramente, lugar sensacional !!!"
165,ChdDSUhNMG9nS0VJQ0FnSUNTbFpYWi1RRRAB,0x94df18c387e4e047:0,5.0,Sensacional! Ótimo atendimento.
212,ChdDSUhNMG9nS0VJQ0FnSURqaWFhNGtRRRAB,0x94df18c51a132009:0,5.0,Tudo SENSACIONAL
332,ChZDSUhNMG9nS0VJQ0FnSUMycWV2VEZnEAE,0x94df18eac4b2643d:0,5.0,"Lugar sensacional, comida maravilhosa ❤️"
555,ChZDSUhNMG9nS0VJQ0FnSUNxLWVyaUJREAE,0x94df192ab2980905:0,5.0,Sensacional
569,ChZDSUhNMG9nS0VJQ0FnSUQwaU56SklBEAE,0x94df192ab2980905:0,5.0,Sensacional
596,ChZDSUhNMG9nS0VJQ0FnSUNBbHZUUGN3EAE,0x94df192ab2980905:0,4.0,Comida sensacional
690,ChZDSUhNMG9nS0VJQ0FnSUNjeThPRFp3EAE,0x94df192acbd036eb:0,5.0,Sensacional.
691,ChZDSUhNMG9nS0VJQ0FnSUNjblpiUkxBEAE,0x94df192acbd036eb:0,5.0,Sensacional.



CLUSTER 9 — 87 reviews


,review_id,place_id,rating,review_text
15,ChZDSUhNMG9nS0VJQ0FnSUM0dWVqd1NBEAE,0x94defd80e5de4f0f:0,5.0,Muito boa
32,Ci9DQUlRQUNvZENodHljRjlvT21wdE1qTjNkMnBtU1Y5Ul...,0x94df18bfe8d9f45f:0,4.0,Comida muito boa
58,ChdDSUhNMG9nS0VJQ0FnSUN4OTV5VmlRRRAB,0x94df18bfe8d9f45f:0,5.0,Comida muito boa
59,ChRDSUhNMG9nS0VJQ0FnSURoOThSRxAB,0x94df18bfe8d9f45f:0,1.0,Muito boa
122,ChdDSUhNMG9nS0VJQ0FnSUNBc3BueHNBRRAB,0x94df18bfe8d9f45f:0,4.0,Boa variedade
123,ChdDSUhNMG9nS0VJQ0FnSURRa0pMbS1RRRAB,0x94df18bfe8d9f45f:0,4.0,Moderno e boa comida
185,Ci9DQUlRQUNvZENodHljRjlvT25GRk9YbFNjeTFXYUdSeF...,0x94df18c51a132009:0,5.0,"Comida muito boa, variedade de pratos quentes ..."
188,Ci9DQUlRQUNvZENodHljRjlvT25GbGFsWnpUV3AxVlY5Sl...,0x94df18c51a132009:0,5.0,Ótimo local boa comida com várias opções
247,ChZDSUhNMG9nS0VJQ0FnSUQwbHM3RGZnEAE,0x94df18c51a132009:0,5.0,Boa comida.
268,ChdDSUhNMG9nS0VJQ0FnSURndjQydTdnRRAB,0x94df18c51a132009:0,4.0,Boa comida


## PCA

In [144]:
from sklearn.decomposition import PCA

# Reduzir o TF-IDF para duas dimensões
pca_2d = PCA(n_components=2)

coordinates_2d = pca_2d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar um DataFrame para a visualização
df_pca_2d = pd.DataFrame({
    "id": preprocessed_reviews_df["review_id"],
    "text": preprocessed_reviews_df["review_text"],
    "place": preprocessed_reviews_df["place_id"],
    "cluster": preprocessed_reviews_df["cluster"].astype(str),
    "PCA1": coordinates_2d[:, 0],
    "PCA2": coordinates_2d[:, 1]
})

# Criar o gráfico
fig = px.scatter(
    df_pca_2d,
    x="PCA1",
    y="PCA2",
    color="cluster",
    hover_name="text",
    hover_data={
        "place": True,
        "PCA1": False,
        "PCA2": False
    },
    title="Clusters das reviews — PCA em 2 dimensões"
)

fig.update_traces(
    marker=dict(size=12)
)

fig.update_layout(
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    legend_title="Cluster"
)

fig.show()

In [145]:
# Reduzir o TF-IDF para três dimensões
pca_3d = PCA(n_components=3)

coordinates_3d = pca_3d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar DataFrame para visualização
df_pca_3d = pd.DataFrame({
    "id": preprocessed_reviews_df["review_id"],
    "text": preprocessed_reviews_df["review_text"],
    "place": preprocessed_reviews_df["place_id"],
    "cluster": preprocessed_reviews_df["cluster"].astype(str),
    "PCA1": coordinates_3d[:, 0],
    "PCA2": coordinates_3d[:, 1],
    "PCA3": coordinates_3d[:, 2]
})

# Criar gráfico 3D
fig = px.scatter_3d(
    df_pca_3d,
    x="PCA1",
    y="PCA2",
    z="PCA3",
    color="cluster",
    hover_name="text",
    hover_data={
        "place": True,
        "PCA1": False,
        "PCA2": False,
        "PCA3": False
    },
    title="Clusters das reviews — PCA em 3 dimensões"
)

fig.update_traces(
    marker=dict(size=6)
)

fig.update_layout(
    scene=dict(
        xaxis_title="Componente principal 1",
        yaxis_title="Componente principal 2",
        zaxis_title="Componente principal 3"
    ),
    legend_title="Cluster"
)

fig.show()

In [146]:
print(
    "Variância explicada em 2D:",
    round(pca_2d.explained_variance_ratio_.sum(), 3)
)

print(
    "Variância explicada em 3D:",
    round(pca_3d.explained_variance_ratio_.sum(), 3)
)

Variância explicada em 2D: 0.094
Variância explicada em 3D: 0.131


## Termos representativos

In [147]:
# Centroides calculados pelo K-Means
centroids = kmeans.cluster_centers_

# Quantidade de termos que queremos mostrar
n_terms = 8

results_clusters = []

for cluster_id in range(N_CLUSTERS):

    # Ordenar os termos pelo peso no centroide
    indices = np.argsort(
        centroids[cluster_id]
    )[::-1]

    principal_indices = indices[:n_terms]

    principal_terms = tfidf_words[principal_indices]
    principal_weights = centroids[
        cluster_id,
        principal_indices
    ]

    results_clusters.append({
        "cluster": cluster_id,
        "representative_tokens": ", ".join(principal_terms)
    })

# Criar DataFrame
df_tokens_clusters = pd.DataFrame(results_clusters)

df_tokens_clusters

,cluster,representative_tokens
0,0,"bom, muito, atendimento, recomendo, lugar, res..."
1,1,"excelente, comida, atendimento, restaurante, l..."
2,2,"comida, muito, atendimento, ótima, deliciosa, ..."
3,3,"maravilhoso, lugar, tudo, restaurante, atendim..."
4,4,"recomendo, saborosa, comida, super, muito, ate..."
5,5,"melhor, blumenau, restaurante, melhores, cidad..."
6,6,"ótimo, atendimento, lugar, comida, excelente, ..."
7,7,"top, demais, lugar, comida, atendimento, ótima..."
8,8,"sensacional, comida, atendimento, lugar, marmi..."
9,9,"boa, comida, muito, atendimento, bom, agradáve..."


## Coesão interna

In [148]:
results_cohesion = []

for cluster_id in sorted(preprocessed_reviews_df["cluster"].unique()):

    # Índices dos documentos pertencentes ao cluster
    indices = preprocessed_reviews_df.index[preprocessed_reviews_df["cluster"] == cluster_id].tolist()

    # Extrair a submatriz de similaridade
    submatrix = similarity_tfidf[np.ix_(indices, indices)]

    # Selecionar apenas a parte superior da matriz,
    # excluindo a diagonal principal
    even_values = submatrix[
        np.triu_indices_from(submatrix, k=1)
    ]

    # Calcular a similaridade média
    mean_cohesion = even_values.mean()

    results_cohesion.append({
        "cluster": cluster_id,
        "n_documents": len(indices),
        "mean_cohesion": mean_cohesion
    })


df_coesao = pd.DataFrame(results_cohesion)

df_coesao

,cluster,n_documents,mean_cohesion
0,0,111,0.683265
1,1,160,0.676899
2,2,1337,0.016110
3,3,69,0.812772
4,4,153,0.145408
5,5,92,0.272843
6,6,113,0.407435
7,7,104,0.825425
8,8,36,0.780491
9,9,87,0.584028


## Distância entre clusters

In [149]:
# Similaridade entre os centróides
similarity_centroids = cosine_similarity(
    kmeans.cluster_centers_
)

# Converter similaridade em distância
distance_centroids = 1 - similarity_centroids

# Criar DataFrame
df_dist_clusters = pd.DataFrame(
    distance_centroids,
    index=[f"Cluster {i}" for i in range(N_CLUSTERS)],
    columns=[f"Cluster {i}" for i in range(N_CLUSTERS)]
)

df_dist_clusters.round(3)

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9
Cluster 0,-0.000,0.982,0.726,0.995,0.810,0.984,0.934,0.981,0.973,0.842
Cluster 1,0.982,0.000,0.748,0.992,0.833,0.970,0.905,0.990,0.968,0.923
Cluster 2,0.726,0.748,-0.000,0.931,0.496,0.844,0.732,0.940,0.903,0.658
Cluster 3,0.995,0.992,0.931,0.000,0.957,0.970,0.968,0.986,0.995,0.996
Cluster 4,0.810,0.833,0.496,0.957,0.000,0.927,0.829,0.943,0.933,0.758
Cluster 5,0.984,0.970,0.844,0.970,0.927,0.000,0.972,0.982,0.990,0.969
Cluster 6,0.934,0.905,0.732,0.968,0.829,0.972,0.000,0.978,0.954,0.923
Cluster 7,0.981,0.990,0.940,0.986,0.943,0.982,0.978,0.000,0.994,0.980
Cluster 8,0.973,0.968,0.903,0.995,0.933,0.990,0.954,0.994,-0.000,0.965
Cluster 9,0.842,0.923,0.658,0.996,0.758,0.969,0.923,0.980,0.965,0.000


In [150]:
results_distances = []

for i in range(N_CLUSTERS):
    for j in range(i + 1, N_CLUSTERS):

        results_distances.append({
            "cluster_1": i,
            "cluster_2": j,
            "distance": distance_centroids[i, j]
        })

df_pairs_clusters = pd.DataFrame(
    results_distances
)

df_pairs_clusters.sort_values("distance")

,cluster_1,cluster_2,distance
18,2,4,0.496238
23,2,9,0.658049
1,0,2,0.726394
20,2,6,0.731629
9,1,2,0.747620
34,4,9,0.757606
3,0,4,0.810430
31,4,6,0.829199
11,1,4,0.832555
8,0,9,0.841662


In [151]:
# Identificar menor e maior distância entre clusters
nearest_pair = df_pairs_clusters.loc[
    df_pairs_clusters["distance"].idxmin()
]

furthest_pair = df_pairs_clusters.loc[
    df_pairs_clusters["distance"].idxmax()
]

# Clusters mais próximos
c1_prox = int(nearest_pair["cluster_1"])
c2_prox = int(nearest_pair["cluster_2"])

# Clusters mais distantes
c1_dist = int(furthest_pair["cluster_1"])
c2_dist = int(furthest_pair["cluster_2"])


print("CLUSTERS MAIS PRÓXIMOS")
print("----------------------")
print(
    f"Cluster {c1_prox} × Cluster {c2_prox}"
)
print(
    f"Distância: {nearest_pair['distance']:.3f}"
)


print("\nCLUSTERS MAIS DISTANTES")
print("-----------------------")
print(
    f"Cluster {c1_dist} × Cluster {c2_dist}"
)
print(
    f"Distância: {furthest_pair['distance']:.3f}"
)

CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 2 × Cluster 4
Distância: 0.496

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 3 × Cluster 9
Distância: 0.996
